# Autopilota su Kaggle — multi-strumento (12 GB RAM + GPU)

Esegue la piattaforma `trading-ai` sul **dataset multi-strumento** sfruttando la RAM di Kaggle: scopre da solo gli strumenti presenti, sceglie il timeframe base e cerca strategie robuste su ciascuno.

## Setup (una volta)
1. **Add Input** → aggiungi il tuo dataset (es. `neuromelo-global-mtf-market-dataset`).
2. (Opzionale) **Settings → Accelerator → GPU**.
3. **Save Version → Schedule** per farlo girare da solo.

Output (report, EA, modelli) in `/kaggle/working`.

In [ ]:
# Installa la piattaforma direttamente dal repo (sempre l'ultima versione su main).
!pip install -q 'git+https://github.com/Mel0mac86/trading-ai.git@main'

## 1) Diagnostica del dataset
Capiamo com'e' fatto: sottocartelle, nomi file e strumenti riconosciuti.
Se il percorso non e' giusto, correggi `DATA` con quello stampato qui sotto.

In [ ]:
from pathlib import Path
from trading_ai.data_engine.sources import discover_instruments

# Percorso del dataset montato. Su Kaggle gli input stanno sotto /kaggle/input.
# Proviamo alcuni percorsi tipici e teniamo il primo che contiene CSV.
CANDIDATI = [
    Path('/kaggle/input/datasets/melomac/neuromelo-global-mtf-market-dataset'),
    Path('/kaggle/input/neuromelo-global-mtf-market-dataset'),
]
DATA = next((p for p in CANDIDATI if p.exists() and any(p.rglob('*.csv'))), None)
if DATA is None:
    # Fallback: cerca ovunque sotto /kaggle/input la cartella con piu' CSV.
    base = Path('/kaggle/input')
    dirs = {}
    for f in base.rglob('*.csv'):
        dirs[f.parent] = dirs.get(f.parent, 0) + 1
    DATA = max(dirs, key=dirs.get).parents[0] if dirs else base
print('DATA =', DATA)

csvs = sorted(DATA.rglob('*.csv'))
print('File CSV trovati:', len(csvs))
for f in csvs[:12]:
    print('  ', f.relative_to(DATA))
print('Strumenti riconosciuti:', discover_instruments(DATA))

## 2) Quali timeframe esistono per ogni strumento
Per non fondere per errore timeframe diversi, rileviamo i token (M1/M5/M15/H1/H4/D1) presenti nei nomi file e scegliamo un **timeframe base** per ciascuno strumento.

In [ ]:
import re
from trading_ai.data_engine.sources import _find_local_files, discover_instruments

TF_TOKENS = ['M1', 'M5', 'M15', 'M30', 'H1', 'H4', 'D1']
# Ordine di preferenza del timeframe BASE da caricare (poi la pipeline ricampiona).
PREFERENZA = ['M15', 'M5', 'M30', 'H1', 'M1', 'H4', 'D1']

strumenti = discover_instruments(DATA)
base_tf = {}
for sym in strumenti:
    files = _find_local_files(sym, DATA)
    presenti = []
    for tf in TF_TOKENS:
        if any(re.search(rf'(?<![A-Z0-9]){tf}(?![0-9])', f.stem.upper()) for f in files):
            presenti.append(tf)
    scelto = next((tf for tf in PREFERENZA if tf in presenti), None)
    base_tf[sym] = scelto
    print(f'{sym:8s} file={len(files):3d}  timeframe presenti={presenti or "(token non nel nome)"}  -> base={scelto}')

## 3) Ricerca strategie robuste su tutti gli strumenti
Per ogni strumento carica il timeframe base e lancia la pipeline completa (scoperta pattern + validazione OOS + Deflated Sharpe + report + EA). Tutto in `/kaggle/working`.

In [ ]:
from trading_ai.data_engine.sources import acquire
from trading_ai.pipeline import run_pipeline, PipelineConfig
from trading_ai.strategy_generator import RiskParams

OUT = Path('/kaggle/working')
risk = RiskParams(sl_atr=2, tp_atr=3, be_atr=1, trail_atr=1.5, max_bars=48)

# Timeframe operativi su cui cercare (la pipeline ricampiona dal base).
TIMEFRAMES = ['H1', 'H4']

riepilogo = []
for sym in strumenti:
    contains = base_tf.get(sym)          # isola il timeframe base nel nome file (se presente)
    raw, src = acquire(sym, datasets_dir=DATA, allow_download=False, contains=contains)
    if src == 'synthetic':
        print(f'[{sym}] nessun dato reale trovato: salto.'); continue
    print(f'[{sym}] fonte={src} candele={len(raw):,} {raw.index.min().date()} -> {raw.index.max().date()}')
    for tf in TIMEFRAMES:
        cfg = PipelineConfig(
            timeframe=tf, horizon=10, n_clusters=25,
            min_frequency=0.001, min_profit_factor=1.1, min_count_test=30,
            use_dsr=True, min_dsr=0.90, risk=risk, instrument=f'{sym}_{tf}',
            reports_dir=OUT/'reports', mql4_dir=OUT/'mql4', mql5_dir=OUT/'mql5',
        )
        try:
            res = run_pipeline(raw, cfg)
            n = len(res.robust_strategies)
        except Exception as e:
            n = -1; print(f'   [{tf}] errore: {str(e)[:120]}')
        riepilogo.append((sym, tf, n))
        if n > 0:
            print(f'   [{tf}] ROBUSTE: {n} -> ' + ', '.join(s.name for s in res.robust_strategies))
        elif n == 0:
            print(f'   [{tf}] nessuna strategia robusta')

print('\n=== RIEPILOGO ===')
for sym, tf, n in riepilogo:
    stato = 'ERRORE' if n < 0 else (f'{n} robuste' if n else '-')
    print(f'  {sym:8s} {tf:3s}  {stato}')
print('\nOutput in /kaggle/working: reports/, mql4/, mql5/')

## 4) (Opzionale) Ottimizzazione robusta (Modulo 7)
Sull'ultimo strumento con una strategia robusta, gira il ciclo di feedback e riesporta l'EA ottimizzato.

In [ ]:
from trading_ai.feedback import FeedbackEngine
from trading_ai.ea_generator import EAGenerator

robuste = [s for sym, tf, n in riepilogo if n > 0]
if 'res' in dir() and res.robust_strategies:
    strat = res.robust_strategies[0]
    fb = FeedbackEngine(min_trades=30)
    out = fb.improve(strat, res.features)
    b, o = out.versions[0].metrics, out.versions[1].metrics
    print('Strategia:', strat.name)
    print('  BASE  -> ret %.2f%%  DD %.2f%%  PF %.2f' % (b['total_return']*100, b['max_drawdown']*100, b['profit_factor']))
    print('  OTTIM -> ret %.2f%%  DD %.2f%%  PF %.2f' % (o['total_return']*100, o['max_drawdown']*100, o['profit_factor']))
    print('  parametri:', out.improved_strategy.risk)
    files = EAGenerator(mql4_dir=OUT/'mql4', mql5_dir=OUT/'mql5').export(out.improved_strategy)
    print('EA ottimizzato:', [str(f) for f in files])
else:
    print('Nessuna strategia robusta da ottimizzare in questa run.')

## (Opzionale) GPU + Scheduling
- **GPU (cuML):** con Accelerator=GPU e RAPIDS, KMeans gira su GPU: utile per alzare `n_clusters` e gli strumenti.
- **Scheduling:** *Save Version → Schedule a notebook run* (settimanale): rigira tutto da solo sui dati aggiornati.